In [ ]:
import cv2
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# Charger CLIP
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Ouvrir la vidéo
video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

frame_count = 0
vectors = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_count += 1
    # Pour alléger, on prend une frame toutes les 30 images (≈1s à 30fps)
    if frame_count % 30 == 0:
        # Convertir BGR (OpenCV) → RGB (PIL)
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # Encodage
        inputs = processor(images=image, return_tensors="pt", padding=True)
        with torch.no_grad():
            vector = model.get_image_features(**inputs).numpy().flatten()

        vectors.append(vector)

cap.release()

print(f"✅ {len(vectors)} vecteurs extraits de la vidéo")


In [ ]:
from moviepy.editor import VideoFileClip
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import torch
import soundfile as sf

# Extraire l’audio
video = VideoFileClip("video.mp4")
video.audio.write_audiofile("audio.wav")

# Charger modèle audio
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

# Charger audio
speech, sr = sf.read("audio.wav")
inputs = processor(speech, sampling_rate=sr, return_tensors="pt")

# Encoder
with torch.no_grad():
    embeddings = model(**inputs).last_hidden_state.mean(dim=1).numpy().flatten()

print(f"✅ Vecteur audio de dimension {len(embeddings)}")
